#  ⭐ `dim_dose_frequencia`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root,  normalize_dose_freq

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet") 
bronze =  bronze[["FREQUENCIA_DOSE"]].drop_duplicates()
bronze.columns

Index(['FREQUENCIA_DOSE'], dtype='object')

In [4]:
bronze.FREQUENCIA_DOSE.nunique()

1576

In [5]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_DOSE_FREQUENCIA.csv", sep=";", index=False)

In [6]:
bronze.head(40)

,FREQUENCIA_DOSE
0,6 hora
1,None
10,cíclico
12,4 hora
14,1 cíclico
16,1 dia
27,1
43,
65,12 hora
71,1 por 1 dia


# 🥈 Silver

normalized data, medium quality


In [7]:
silver = normalize_dose_freq(bronze, col="FREQUENCIA_DOSE")
silver.head()

,FREQUENCIA_DOSE,FREQUENCIA_DOSE_DOSES,FREQUENCIA_DOSE_INTERVALO_VALOR,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
0,6 hora,1.0,6.0,hora,3
1,None,NaN,NaN,desconhecido,0
10,cíclico,1.0,NaN,ciclico,9
12,4 hora,1.0,4.0,hora,3
14,1 cíclico,1.0,NaN,ciclico,9


In [8]:
silver[['FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE', 'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR']].drop_duplicates().sort_values(by='FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR')

,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
1,desconhecido,0
4545,segundo,1
4432,minuto,2
0,hora,3
16,dia,4
1037,semana,5
566,mes,6
19443,ano,7
281969,decada,8
10,ciclico,9


In [9]:
silver.FREQUENCIA_DOSE_INTERVALO_VALOR.value_counts(dropna=False).head(10)

FREQUENCIA_DOSE_INTERVALO_VALOR
1.0     230
2.0     117
8.0     114
NaN     111
6.0     101
3.0      96
4.0      95
12.0     78
21.0     48
15.0     45
Name: count, dtype: int64

In [10]:
silver.FREQUENCIA_DOSE_DOSES.value_counts(dropna=False).head(10)

FREQUENCIA_DOSE_DOSES
1.0     529
2.0     124
3.0      91
4.0      83
6.0      63
5.0      51
8.0      38
12.0     37
7.0      31
10.0     29
Name: count, dtype: int64

In [11]:
silver.query("FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE == 'desconhecido'").FREQUENCIA_DOSE.value_counts(dropna=False).head(40)

FREQUENCIA_DOSE
None                      1
15                        1
2 por 12                  1
1 por 01 se necessário    1
24                        1
86                        1
7 por 28                  1
4 por 1                   1
1                         1
0.2                       1
01 por 01                 1
07                        1
40                        1
300                       1
20                        1
0 se necessário           1
11                        1
18 se necessário          1
0.0 se necessário         1
1.0 se necessário         1
01 por 1                  1
2 por 1                   1
10 se necessário          1
4 por 3 se necessário     1
1 por 2                   1
1 por 4 se necessário     1
1 por 24                  1
325                       1
2 por 7                   1
08                        1
130                       1
02 se necessário          1
400                       1
4 por 4 se necessário     1
06 se necessário          1
100 

In [12]:
hist = silver

In [13]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_dose_frequencia/hist_dim_dose_frequencia.parquet", index=False)

In [14]:
hist.to_csv(Path(project_root) / "data/02_silver/hist_dim_dose_frequencia/hist_dim_dose_frequencia_MANUAL.csv", sep=";",index=False)

# 🥇 Gold


In [15]:
hist.columns

Index(['FREQUENCIA_DOSE', 'FREQUENCIA_DOSE_DOSES',
       'FREQUENCIA_DOSE_INTERVALO_VALOR',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR'],
      dtype='object')

In [16]:
dim = hist[['FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE',
       'FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR']].sort_values(by='FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR').drop_duplicates()
dim

,FREQUENCIA_DOSE_INTERVALO_TIPO_CHAVE,FREQUENCIA_DOSE_INTERVALO_TIPO_VALOR
550041,desconhecido,0
109395,segundo,1
204825,minuto,2
401527,hora,3
329381,dia,4
253533,semana,5
529128,mes,6
232746,ano,7
281969,decada,8
277196,ciclico,9


In [17]:
dim.to_parquet(Path(project_root) / "data/03_gold/dim_dose_frequencia/dim_dose_frequencia.parquet", index=False)